# M0.3 · The autonomy ladder

**Module 0 — the shared core → The Shared Core**  ·  *Both directions*

---

**Risk.** Workflows are promoted to autonomy on vibes, then can't be demoted.

**Control.** L1→L2→L2.5→L3 with named promotion evidence and a named demotion authority.

**This lab.** Walk one workflow up the autonomy ladder with named promotion evidence.

| | |
|---|---|
| Open-source tooling | kagent |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("M0.3"))

The autonomy ladder is not about model capability. It is about what the model's output is allowed to trigger without a human in the path. A tiny model at L3 is more dangerous than a frontier model at L1.

Read the rungs, then test a claim against reality.

In [ ]:
from cybercommons import planes
print(planes.describe_ladder())

Every team says it operates at L2. Check four manifests that all *claim* a rung and see which ones can support the claim.

In [ ]:
W = planes.Tool
claims = [
    planes.Manifest("doc-summariser", [W("read_file")], rung="L1"),
    planes.Manifest("pr-commenter",
                    [W("read_file"), W("post_comment", writes=True, scope="project")],
                    rung="L2"),
    planes.Manifest("patch-bot",
                    [W("read_file"), W("write_file", writes=True, scope="project")],
                    approval_required={"write_file"}, rung="L2"),
    planes.Manifest("remediator",
                    [W("read_file"), W("deploy_prod", writes=True, scope="org",
                                       reversible=False)],
                    rung="L2.5"),
]
for m in claims:
    problems = m.rung_check()
    verdict = "consistent" if not problems else "CLAIM NOT SUPPORTED"
    print(f"{m.agent:16s} claims {m.rung:5s} → {verdict}")
    for p in problems:
        print(f"    · {p}")

### Expect

`doc-summariser` and `patch-bot` are consistent — the first holds no writer, the second gates the one it has. `pr-commenter` claims L2 with an ungated writer. `remediator` claims L2.5 while holding an org-wide irreversible tool with no gate.

### Your turn

Pick a real agent in your organisation. Write out its manifest honestly — every tool, every scope — and run `rung_check()`. The usual result is that the claimed rung is one to two rungs below the one the controls actually support.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/M0.3.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*